In [6]:
# Data was exported from R so it can be used in python

# Load and Inspect Data in python

import pandas as pd

df = pd.read_csv("classifier_input.csv", index_col = 0)
print(df.shape)
df.head()


(12, 31)


,MT4,CTHRC1,CD163,CD300E,CST6,IGFL2,GSTO1,COL4A1,SERPINE1,SERPINB7,...,CD276,COL4A2,TOB2,ARHGEF26,LTF,PSAPL1,EIF4EBP1,PI15,PER2,condition
donor1_lesional,6.125105,8.229915,6.720274,6.650569,8.624690,5.799774,10.782634,8.600247,8.395316,5.923145,...,8.402402,8.261433,10.534351,7.661204,9.954703,4.597461,10.089223,7.053035,8.695626,lesional
donor1_nonlesional,9.298202,6.400012,5.057577,4.998058,9.441933,8.432811,10.026204,7.166627,7.308127,8.592244,...,7.344425,6.816526,11.060483,8.694212,7.996533,7.425730,9.182659,4.100259,9.761210,nonlesional
donor2_lesional,4.057701,9.595600,7.027818,6.568600,8.307901,5.583720,10.668422,10.014147,9.800563,4.481870,...,7.941073,9.707868,10.095207,7.412238,9.066208,3.787318,9.608806,7.855020,8.967138,lesional
donor2_nonlesional,7.659820,8.667997,4.590448,4.590448,10.580854,7.636092,10.080816,9.099603,8.725344,7.815811,...,7.191268,8.440336,10.880044,8.169893,7.374508,6.383176,8.714055,6.594914,9.757022,nonlesional
donor3_lesional,2.552301,10.443553,9.179952,8.331524,7.970233,5.680291,11.356444,10.799155,10.393983,4.991683,...,9.321990,10.387010,9.523012,6.041679,9.183981,4.354929,10.289423,9.250799,7.922808,lesional


In [7]:
# Separate features from the label

x = df.drop(columns= ["condition"])
y = df["condition"]

print(x.shape)
y.value_counts()


(12, 30)


condition
lesional       6
nonlesional    6
Name: count, dtype: int64

In [10]:
# Setup Leave-One-Out-Cross-Validation with logistic regression

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score

leoo = LeaveOneOut()
model = LogisticRegression(max_iter = 1000)

scores = cross_val_score(model, x, y, cv = leoo)
scores



array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [11]:
# Load the full expression data

df_full = pd.read_csv("classifier_input_full.csv", index_col=0)
df_full.shape

(12, 29391)

In [13]:
# Separate features and labels again

x_full = df_full.drop(columns = ["condition"])
y_full = df_full["condition"]

print(x_full.shape)
print(y_full.value_counts())

(12, 29390)
condition
lesional       6
nonlesional    6
Name: count, dtype: int64


In [14]:
# Build a leak-free pipeline

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("feature_selection", SelectKBest(score_func = f_classif, k=30)),
    ("classifier", LogisticRegression(max_iter = 1000))
])

scores_full = cross_val_score(pipeline, x_full, y_full, cv = leoo)
scores_full

C:\Users\adeol\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [    2    93   229   230   308   309   384   555   674   771   895   952
   983  1150  1239  1270  1362  1399  1896  1919  2223  2514  2527  2741
  3043  3097  3330  3332  3478  3481  3881  3895  3986  3987  3990  4035
  4103  4154  4155  4488  4677  4865  4990  5031  5042  5083  5175  5208
  5226  5300  5376  5553  5618  5627  5699  5710  5744  5908  6003  6147
  6149  6218  6219  6349  6352  6394  6403  6428  6548  6556  6865  6894
  6955  6964  7124  7141  7311  7325  7386  7509  7520  8075  8106  8109
  8138  8174  8622  8631  8633  8637  9062  9096  9187  9276  9456  9534
  9794  9840  9904  9954  9967 10069 10073 10283 10393 10454 10463 10518
 10576 10602 10621 10623 10729 10939 10984 10994 11010 11089 11193 11339
 11462 11551 11707 11716 11930 11934 12445 12542 12649 12946 12992 13273
 13326 13327 13331 13332 13438 13485 13499 13633 13643 13676 13687 13708
 14

array([0., 0., 1., 1., 1., 0., 1., 1., 0., 1., 1., 1.])

In [16]:
# Calculate mean accuracy and look at which samoles were misclassified

print("Mean Accuracy:", scores_full.mean())

misclassified = pd.DataFrame({
    "sample" : x_full.index,
    "true_label" : y_full.values,
    "correct" : scores_full
})

misclassified
    

Mean Accuracy: 0.6666666666666666


,sample,true_label,correct
0,donor1_lesional,lesional,0.0
1,donor1_nonlesional,nonlesional,0.0
2,donor2_lesional,lesional,1.0
3,donor2_nonlesional,nonlesional,1.0
4,donor3_lesional,lesional,1.0
5,donor3_nonlesional,nonlesional,0.0
6,donor4_lesional,lesional,1.0
7,donor4_nonlesional,nonlesional,1.0
8,donor5_lesional,lesional,0.0
9,donor5_nonlesional,nonlesional,1.0
